<div align="center">

# **DecisionTree Project Notebook**

</div>

### **Notebook Purpose**
This notebook shows how to build a Decision Tree from scratch. You will learn how to create, train and test a tree, and understand things like entropy, Gini, how the tree creates and works, and how to make it simpler using pruning.

### **Learning Goals**

* Build the Decision Tree and Node classes with full features
* Learn and code how to split data using Gini and Entropy
* Understand how the tree makes predictions and moves through nodes
* Use pruning to make the model simpler and better
* Check your code with clear and complete unit tests

### **Tasks**
Finish all the TODO parts in this notebook to create a Decision Tree classifier and pass all the tests.

# **Libraries Imports**
Import required libraries and add the DecisionTree and Node classes.

In [41]:
# ============================================================================
# LIBRARY IMPORTS
# ============================================================================

import numpy as np
import pandas as pd
import unittest
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from graphviz import Digraph

import networkx as nx
import matplotlib.pyplot as plt
# Set random seed for reproducibility
np.random.seed(42)

# ============================================================================
# DECISION TREE CLASSES IMPORT
# ============================================================================

from DT_Library import Node
from DT_Library import DecisionTree

## **Part 1 — Node Class**

### **What is a Node?**

A Node is one part of the decision tree. It stores the feature to split on, its children, and the final value prediction.  
(It can also be a leaf node.)

The Node class with TODOs is in [DT_Library](DT_Library.py) — complete it and tests will check if it works correctly.


In [42]:
# ============================================================================
# TESTS FOR NODE CLASS
# ============================================================================

class TestNodeInit(unittest.TestCase):
    
    def test_default_initialization(self):
        # Test default constructor behavior
        node = Node()
        self.assertTrue(hasattr(node, 'feature'))
        self.assertTrue(hasattr(node, 'children'))
    
    def test_feature_assignment(self):
        # Test feature parameter assignment
        node = Node(feature="age")
        self.assertEqual(node.feature, "age")
        self.assertIsNone(node.children)
    
    def test_children_assignment(self):
        # Test children parameter assignment
        child_list = [Node(), Node()]
        node = Node(children=child_list)
        self.assertEqual(len(node.children), 2)
        self.assertIsInstance(node.children[0], Node)

# Run the tests
runner = unittest.TextTestRunner(verbosity=2)
suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestNodeInit)
result = runner.run(suite)

print("\nAll tests passed." if result.wasSuccessful() else "\nSome tests failed.")

test_children_assignment (__main__.TestNodeInit.test_children_assignment) ... ok
test_default_initialization (__main__.TestNodeInit.test_default_initialization) ... ok
test_feature_assignment (__main__.TestNodeInit.test_feature_assignment) ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.004s

OK



All tests passed.


## **Part 2 — DecisionTree Hyperparameters**

### **Hyperparameters**

Hyperparameters control how the tree learns from data.

* **max_Depth** – maximum levels the tree can grow. A shallow tree may **underfit**, a very deep tree may **overfit**.
* **min_Samples** – minimum samples required to split a node.
* **pruning_threshold** – limits how small improvements must be to keep splitting. Helps reduce overfitting.
* **mode** – choice of splitting method (Gini or Entropy). Different methods can affect tree decisions.

You can add other hyperparameters if needed to improve your class.

A DecisionTree class with TODOs is in [DT_Library](DT_Library.py) — complete it and tests will check if it handles hyperparameters correctly.


In [43]:
# ============================================================================
# TESTS FOR DECISIONTREE CONSTRUCTOR
# ============================================================================

class TestDecisionTreeInit(unittest.TestCase):
    
    def test_constructor_attributes_exist(self):
        # Check required attributes are created
        dt = DecisionTree()
        self.assertTrue(hasattr(dt, 'mode'))
        self.assertTrue(hasattr(dt, 'max_Depth'))
        self.assertTrue(hasattr(dt, 'min_Samples'))
        self.assertTrue(hasattr(dt, 'pruning_threshold'))
        self.assertTrue(hasattr(dt, 'root'))
    
    def test_default_values_assignment(self):
        # Verify default parameter values are stored correctly
        dt = DecisionTree()
        self.assertEqual(dt.max_Depth, float("inf"))
        self.assertIsNone(dt.root)
    
    def test_custom_parameters_override_defaults(self):
        # Custom values should replace defaults
        dt = DecisionTree(mode="gini", max_Depth=10, min_Samples=5, pruning_threshold=0.01)
        self.assertEqual(dt.mode, "gini")
        self.assertEqual(dt.max_Depth, 10)
        self.assertEqual(dt.min_Samples, 5)
        self.assertEqual(dt.pruning_threshold, 0.01)
        self.assertIsNone(dt.root)

# Run the tests
runner = unittest.TextTestRunner(verbosity=2)
suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestDecisionTreeInit)
result = runner.run(suite)

print("\nAll tests passed." if result.wasSuccessful() else "\nSome tests failed.")

test_constructor_attributes_exist (__main__.TestDecisionTreeInit.test_constructor_attributes_exist) ... ok
test_custom_parameters_override_defaults (__main__.TestDecisionTreeInit.test_custom_parameters_override_defaults) ... ok
test_default_values_assignment (__main__.TestDecisionTreeInit.test_default_values_assignment) ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.004s

OK



All tests passed.


## **Part 3 — Splitting Criteria**

### **How the Tree Chooses Splits**

This part shows how the tree decides the best feature to split. We use measures like **Gini** and **Information Gain** to guide these decisions.

## **Part 3.1 — Entropy & Information Gain**

### **What is Entropy and Information Gain?**

* **Entropy** measures how mixed the classes are. For a target variable (Y) with classes (c):

$$Entropy(Y) = -\sum_{c} p(c) \log p(c)$$

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; **NOTE:** p(c) is the probability of class (c).

* **Conditional Entropy** shows the remaining uncertainty after splitting by a feature (X) with values (v):

$$Entropy(Y|X) = \sum_{v} P(X=v) \cdot Entropy(Y|X=v)$$

* **Information Gain** tells how much uncertainty is reduced by splitting on a feature:

$$GAIN(Y, X) = Entropy(Y) - Entropy(Y|X)$$

* Entropy is high when classes are mixed equally, low when one class dominates
* Information Gain shows how useful a feature is for reducing uncertainty
* *GAIN(Y, X) = 0* -> feature gives no information
* *GAIN(Y, X) = Entropy(Y)* -> feature perfectly separates classes

*Entropy* and *Information Gain* functions with TODOs are in [DT_Library](DT_Library.py) — complete them so that the tests check your functions return correctly.


In [44]:
# ============================================================================
# TESTS FOR INFORMATION GAIN
# ============================================================================

class TestInformationGain(unittest.TestCase):
   
    def test_returns_numeric_value(self):
        # Must return a number not None
        Y_simple = np.array([0, 1, 0, 1, 1, 0])
        X_simple = pd.DataFrame({
            'feature_simple': [1, 5, 3, 5, 3, 1],
            'other': [1, 2, 4, 4, 5, 1]
        })
        feature_simple = X_simple['feature_simple']

        dt = DecisionTree()

        result = dt._information_Gain(feature_simple, X_simple, Y_simple)
        self.assertIsNotNone(result, "Method should return a value, not None")
        self.assertIsInstance(result, (int, float, np.number), "Result must be numeric")
        self.assertTrue(np.isfinite(result), "Result must be finite")
    
    def test_constant_feature_zero_gain(self):
        # Constant feature should give zero information gain
        Y_mixed = np.array([0, 1, 0, 1, 1, 0])
        X_constant = pd.DataFrame({
            'constant_feature': [5, 5, 5, 5, 5, 5],
            'other': [1, 2, 3, 4, 5, 6]
        })
        feature_constant = X_constant['constant_feature']
        
        dt = DecisionTree()

        result = dt._information_Gain(feature_constant, X_constant, Y_mixed)
        self.assertIsInstance(result, (int, float, np.number))
        self.assertAlmostEqual(result, 0.0, places=10, msg="Constant feature should give zero information gain")
    
    def test_pure_classes_zero_or_low_gain(self):
        # When Y is pure (all same class), gain should be low
        Y_pure = np.array([1, 1, 1, 1])
        X_any = pd.DataFrame({
            'any_feature': [0, 0, 1, 1],
            'other': [10, 20, 30, 40]
        })
        feature_any = X_any['any_feature']

        dt = DecisionTree()
        
        result = dt._information_Gain(feature_any, X_any, Y_pure)
        self.assertIsInstance(result, (int, float, np.number))
        self.assertLessEqual(result, 1e-10, "Pure classes should give very low information gain")
    
    def test_exact_information_gain_calculation(self):
        # Test with exact computable values
        Y_test = np.array([0, 0, 1, 1, 0, 0, 0])
        X_test = pd.DataFrame({
            'perfect_split': [0, 0, 1, 0, 2, 2, 2],
            'other': [10, 20, 30, 40, 10, 5, 15]
        })
        feature_test = X_test['perfect_split']

        dt = DecisionTree()

        result = dt._information_Gain(feature_test, X_test, Y_test)

        self.assertIsInstance(result, (int, float, np.number))
        self.assertAlmostEqual(result, 0.46956521111470667, places=3, msg=f"That {result} is not valid!")

# Run the tests
runner = unittest.TextTestRunner(verbosity=2)
suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestInformationGain)
result = runner.run(suite)

print("\nAll tests passed." if result.wasSuccessful() else "\nSome tests failed.")

test_constant_feature_zero_gain (__main__.TestInformationGain.test_constant_feature_zero_gain) ... ok
test_exact_information_gain_calculation (__main__.TestInformationGain.test_exact_information_gain_calculation) ... ok
test_pure_classes_zero_or_low_gain (__main__.TestInformationGain.test_pure_classes_zero_or_low_gain) ... ok
test_returns_numeric_value (__main__.TestInformationGain.test_returns_numeric_value) ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.011s

OK



All tests passed.


## **Part 3.2 — Gini Impurity & Gini Split**

### **What is Gini Impurity and Gini Split?**

* **Gini Impurity** measures how impure or mixed the classes are. For a target variable (Y) with classes (c):

$$Gini(Y) = 1 - \sum_{c} p(c)^2$$

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; **NOTE:** p(c) is the probability of class (c).

* **Gini Split** calculates the weighted impurity after splitting by a feature (X) with values (v):

$$Gini\_Split(Y, X) = \sum_{v} \frac{|Y_v|}{|Y|} \cdot Gini(Y_v)$$

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; **NOTE:** |Y_v| is the number of samples where X=v, and Y_v are the corresponding labels.

* Gini ranges from 0 (pure) to 0.5 (maximum impurity for binary classes)
* Lower Gini Split values indicate better splits

*Gini Impurity* and *Gini Split* functions with TODOs are in [DT_Library](DT_Library.py) — complete them so that the tests check your functions return correctly.

In [45]:
# ============================================================================
# TESTS FOR GINI SPLIT
# ============================================================================

class TestGiniSplit(unittest.TestCase):
   
    def test_returns_numeric_value(self):
        # Must return a number not None
        Y_simple = np.array([0, 1, 0, 1, 1, 0])
        X_simple = pd.DataFrame({
            'feature_simple': [1, 5, 3, 5, 3, 1],
            'other': [1, 2, 4, 4, 5, 1]
        })
        feature_simple = X_simple['feature_simple']

        dt = DecisionTree()

        result = dt._gini_Split(feature_simple, X_simple, Y_simple)
        self.assertIsNotNone(result, "Method should return a value, not None")
        self.assertIsInstance(result, (int, float, np.number), "Result must be numeric")
        self.assertTrue(np.isfinite(result), "Result must be finite")
    
    def test_constant_feature_original_gini(self):
        # Constant feature should return original gini impurity
        Y_mixed = np.array([0, 1, 0, 1, 1, 0])
        X_constant = pd.DataFrame({
            'constant_feature': [5, 5, 5, 5, 5, 5],
            'other': [1, 2, 3, 4, 5, 6]
        })
        feature_constant = X_constant['constant_feature']
        
        dt = DecisionTree()

        result = dt._gini_Split(feature_constant, X_constant, Y_mixed)
        # Original Gini for [0,1,0,1,1,0]: 1 - (3/6)^2 - (3/6)^2 = 0.5
        self.assertIsInstance(result, (int, float, np.number))
        self.assertAlmostEqual(result, 0.5, places=10, msg="Constant feature should return original gini impurity")
    
    def test_pure_classes_zero_gini(self):
        # When Y is pure (all same class), gini should be zero
        Y_pure = np.array([1, 1, 1, 1])
        X_any = pd.DataFrame({
            'any_feature': [0, 0, 1, 1],
            'other': [10, 20, 30, 40]
        })
        feature_any = X_any['any_feature']

        dt = DecisionTree()
        
        result = dt._gini_Split(feature_any, X_any, Y_pure)
        self.assertIsInstance(result, (int, float, np.number))
        self.assertLessEqual(result, 1e-10, "Pure classes should give zero gini impurity")
    
    def test_exact_gini_split_calculation(self):
        # Test with exact computable values
        Y_test = np.array([0, 0, 1, 1, 0, 0, 0])
        X_test = pd.DataFrame({
            'split_feature': [0, 0, 1, 0, 2, 2, 2],
            'other': [10, 20, 30, 40, 10, 5, 15]
        })
        feature_test = X_test['split_feature']

        dt = DecisionTree()

        result = dt._gini_Split(feature_test, X_test, Y_test)

        self.assertIsInstance(result, (int, float, np.number))
        self.assertAlmostEqual(result, 0.19047619047619047, places=3, msg=f"That {result} is not valid!")

# Run the tests
runner = unittest.TextTestRunner(verbosity=2)
suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestGiniSplit)
result = runner.run(suite)

print("\nAll tests passed." if result.wasSuccessful() else "\nSome tests failed.")

test_constant_feature_original_gini (__main__.TestGiniSplit.test_constant_feature_original_gini) ... ok
test_exact_gini_split_calculation (__main__.TestGiniSplit.test_exact_gini_split_calculation) ... ok
test_pure_classes_zero_gini (__main__.TestGiniSplit.test_pure_classes_zero_gini) ... ok
test_returns_numeric_value (__main__.TestGiniSplit.test_returns_numeric_value) ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.010s

OK



All tests passed.


## **Part 3.3 — Best Feature Selection**

### **What is Best Feature Selection?**

This process finds the feature that gives the best split according to the chosen criterion (Gini or Information Gain).  
Calculate splitting criterion for each feature in the dataset and then compare all features to select the feature with the best score.

- For **Information Gain** higher values are better
- For **Gini Split** lower values are better
- The function should return a Node object containing the best feature information

*Best Feature Selection* function with TODOs is in [DT_Library](DT_Library.py) — complete it so that the tests check your function works correctly with both Gini and Information Gain modes.

In [46]:
# ============================================================================
# TESTS FOR BEST FEATURE SELECTION
# ============================================================================

class TestBestFeature(unittest.TestCase):
   
    def test_returns_node_object(self):
        # Must return a Node object not None
        Y_simple = np.array([0, 1, 0, 1, 1, 0])
        X_simple = pd.DataFrame({
            'feature1': [1, 5, 3, 5, 3, 1],
            'feature2': [2, 3, 1, 3, 1, 2]
        })

        dt = DecisionTree(mode="gain")
        result = dt._get_best_Feature(X_simple, Y_simple)
        
        self.assertIsNotNone(result, "Method should return a Node, not None")
        self.assertIsInstance(result, Node, "Result must be a Node object")
    
    def test_gain_mode_selects_best_feature(self):
        # Gain mode should select feature with highest information gain
        Y_test = np.array([0, 0, 1, 1, 0, 0])
        X_test = pd.DataFrame({
            'good_feature': [0, 0, 1, 1, 0, 0],  # perfect correlation
            'bad_feature': [1, 1, 1, 1, 1, 1]   # constant feature
        })
        
        dt = DecisionTree(mode="gain")
        result = dt._get_best_Feature(X_test, Y_test)
        
        self.assertIsInstance(result, Node)
        # Should select the good feature that provides information
        self.assertIsNotNone(result.feature, "Selected feature should not be None")
    
    def test_gini_mode_selects_best_feature(self):
        # Gini mode should select feature with lowest gini split
        Y_test = np.array([0, 0, 1, 1, 0, 0])
        X_test = pd.DataFrame({
            'good_feature': [0, 0, 1, 1, 0, 0],  # good separation
            'bad_feature': [2, 2, 2, 2, 2, 2]   # constant feature
        })
        
        dt = DecisionTree(mode="gini")
        result = dt._get_best_Feature(X_test, Y_test)
        
        self.assertIsInstance(result, Node)
        # Should select the good feature that reduces impurity
        self.assertIsNotNone(result.feature, "Selected feature should not be None")
    
    def test_different_modes_same_data(self):
        # Both modes should work on same dataset
        Y_test = np.array([0, 1, 0, 1, 1, 0, 0])
        X_test = pd.DataFrame({
            'feature_a': [1, 2, 1, 2, 1, 2, 1],
            'feature_b': [5, 5, 3, 3, 5, 3, 5]
        })
        
        dt_gain = DecisionTree(mode="gain")
        dt_gini = DecisionTree(mode="gini")
        
        result_gain = dt_gain._get_best_Feature(X_test, Y_test)
        result_gini = dt_gini._get_best_Feature(X_test, Y_test)
        
        self.assertIsInstance(result_gain, Node)
        self.assertIsInstance(result_gini, Node)
        # Both should return valid Node objects
        self.assertTrue(hasattr(result_gain, 'feature'))
        self.assertTrue(hasattr(result_gini, 'feature'))

# Run the tests
runner = unittest.TextTestRunner(verbosity=2)
suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestBestFeature)
result = runner.run(suite)

print("\nAll tests passed." if result.wasSuccessful() else "\nSome tests failed.")

test_different_modes_same_data (__main__.TestBestFeature.test_different_modes_same_data) ... ok
test_gain_mode_selects_best_feature (__main__.TestBestFeature.test_gain_mode_selects_best_feature) ... ok
test_gini_mode_selects_best_feature (__main__.TestBestFeature.test_gini_mode_selects_best_feature) ... ok
test_returns_node_object (__main__.TestBestFeature.test_returns_node_object) ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.011s

OK



All tests passed.


## **Part 4 — Tree Building & Training**

### **How the Tree Learns**

You will build the full decision tree by splitting data step by step until stopping rules are met.

- **fit()** initializes the training process and sets up the root node.
- **_create_Tree()** recursively builds tree structure by finding best splits.
- The process stops when the tree reaches its maximum depth, when there are too few samples to split, or when all samples in a node belong to one class.
- Each node saves the best feature and links to its children for different feature values.

*fit* and *_create_Tree* functions with TODOs are in [DT_Library](DT_Library.py) — complete them so that the tests check your functions build valid tree structures correctly.

In [47]:
# ============================================================================
# TESTS FOR TREE BUILDING & TRAINING
# ============================================================================

class TestTreeBuilding(unittest.TestCase):

    # ===============================
    # HELPER FUNCTIONS
    # ===============================

    def calc_depth(self, node):
        if node is None or not hasattr(node, "children") or not node.children:
            return 0
        return 1 + max(self.calc_depth(child) for child in node.children)

    def count_nodes(self, node):
        if node is None:
            return 0
        if not hasattr(node, "children") or not node.children:
            return 1
        return 1 + sum(self.count_nodes(child) for child in node.children)
    
    # ===============================
    # TEST FUNCTIONS
    # ===============================

    def test_fit_creates_root_node(self):
        # Check if fit() creates a valid root node
        Y_simple = np.array([0, 0, 0, 0, 1, 1, 1, 2])
        X_simple = pd.DataFrame({
            'sequential': [1, 2, 3, 4, 5, 6, 7, 8],  # simple increasing values
            'noise': [10, 11, 12, 13, 14, 15, 16, 17]
        })

        dt = DecisionTree(mode="gain")
        dt.fit(X_simple, Y_simple)
        
        self.assertIsNotNone(dt.root, "Root node should exist after training")
        self.assertIsInstance(dt.root, Node)
        self.assertIsNotNone(dt.root.feature, "Root should choose a feature to split on")
        self.assertIn(dt.root.feature, X_simple.columns, "Root feature must be one of the dataset columns")
        
    def test_perfectly_balanced_tree(self):
        # Check if a balanced dataset creates a balanced binary tree
        Y_balanced = np.array([0, 0, 1, 1, 0, 0, 1, 1])
        X_balanced = pd.DataFrame({
            'binary_split': [0, 0, 1, 1, 0, 0, 1, 1],
            'secondary': [0, 1, 0, 1, 1, 0, 1, 0]
        })
        
        dt_gain = DecisionTree(mode="gain", max_Depth=3)
        dt_gini = DecisionTree(mode="gini", max_Depth=3)

        dt_gain.fit(X_balanced, Y_balanced)
        dt_gini.fit(X_balanced, Y_balanced)

        self.assertEqual(dt_gain.root.feature, 'binary_split', "Should pick the best splitting feature")
        self.assertEqual(dt_gini.root.feature, 'binary_split', "Should pick the best splitting feature")
    
    def test_early_stopping_min_samples(self):
        # Check that min_Samples stops further splitting
        Y = np.array([0, 0, 1, 1, 2, 2, 1, 1])
        X = pd.DataFrame({
            'feature1': [0, 0, 1, 1, 0, 0, 1, 1],
            'feature2': [0, 0, 0, 1, 1, 1, 1, 0]
        })

        dt_strict = DecisionTree(mode="gain", min_Samples=5)
        dt_lenient = DecisionTree(mode="gain", min_Samples=2)

        dt_strict.fit(X, Y)
        dt_lenient.fit(X, Y)

        depth_strict = self.calc_depth(dt_strict.root)
        depth_lenient = self.calc_depth(dt_lenient.root)

        self.assertEqual(depth_strict, 1, "Tree with high min_Samples should grow less")
        self.assertEqual(depth_lenient, 2, "Tree with small min_Samples should grow deeper")

    def test_depth_limited_vs_unlimited(self):
        # Check depth limit works correctly
        Y = np.array([0, 1, 2, 1, 1, 3, 1, 1, 3, 0, 1, 2])
        X = pd.DataFrame({
            "feature1": [1, 2, 2, 2, 2, 2, 3, 3, 3, 1, 4, 4],
            "feature2": [0, 1, 2, 0, 1, 3, 0, 1, 3, 0, 1, 1],
            'feature3': [0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1],
            'feature4': [0, 2, 0, 1, 1, 2, 1, 0, 0, 1, 2, 1]
        })

        dt_limited = DecisionTree(mode="gini", max_Depth=2)
        dt_unlimited = DecisionTree(mode="gini", max_Depth=float("inf"))

        dt_limited.fit(X, Y)
        dt_unlimited.fit(X, Y)

        depth_limited = self.calc_depth(dt_limited.root)
        depth_unlimited = self.calc_depth(dt_unlimited.root)

        self.assertEqual(depth_limited, 2, "Tree must not go deeper than max_Depth")
        self.assertGreaterEqual(depth_unlimited, depth_limited, "Unlimited tree should grow at least as deep")

    def test_node_count_comparison(self):
        # Compare number of nodes for different constraints
        Y = np.array([0, 1, 2, 1, 1, 3, 1, 1, 3, 0, 1, 2])
        X = pd.DataFrame({
            "feature1": [1, 2, 2, 2, 2, 2, 3, 3, 3, 1, 4, 4],
            "feature2": [0, 1, 2, 0, 1, 3, 0, 1, 3, 0, 1, 1],
            'feature3': [0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1],
            'feature4': [0, 2, 0, 1, 1, 2, 1, 0, 0, 1, 2, 1]
        })

        dt_constrained = DecisionTree(mode="gain", max_Depth=2, min_Samples=5)
        dt_free = DecisionTree(mode="gain", max_Depth=3, min_Samples=2)

        dt_constrained.fit(X, Y)
        dt_free.fit(X, Y)

        n_constrained = self.count_nodes(dt_constrained.root)
        n_free = self.count_nodes(dt_free.root)

        self.assertGreaterEqual(n_free, n_constrained, f"Tree with fewer limits ({n_free}) should have at least as many nodes as constrained one ({n_constrained})")
        self.assertEqual(n_free, 13, "Free tree should have more nodes, including a full root and children")

    def test_single_feature_dominance(self):
        # Check that the tree picks the best possible feature
        Y_dominant = np.array([0, 0, 0, 1, 1, 1, 2, 2])
        X_dominant = pd.DataFrame({
            'perfect_feature': [1, 1, 1, 2, 2, 2, 3, 3],
            'random_feature': [9, 3, 7, 1, 5, 8, 2, 6],
            'constant_feature': [5, 5, 5, 5, 5, 5, 5, 5]
        })
        
        dt = DecisionTree(mode="gain", max_Depth=5)
        dt.fit(X_dominant, Y_dominant)
        
        self.assertEqual(dt.root.feature, 'perfect_feature', "Tree should pick the feature that perfectly matches the target")

# Run the tests
runner = unittest.TextTestRunner(verbosity=2)
suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestTreeBuilding)
result = runner.run(suite)

print("\nAll tests passed." if result.wasSuccessful() else "\nSome tests failed.")

test_depth_limited_vs_unlimited (__main__.TestTreeBuilding.test_depth_limited_vs_unlimited) ... ok
test_early_stopping_min_samples (__main__.TestTreeBuilding.test_early_stopping_min_samples) ... ok
test_fit_creates_root_node (__main__.TestTreeBuilding.test_fit_creates_root_node) ... ok
test_node_count_comparison (__main__.TestTreeBuilding.test_node_count_comparison) ... ok
test_perfectly_balanced_tree (__main__.TestTreeBuilding.test_perfectly_balanced_tree) ... ok
test_single_feature_dominance (__main__.TestTreeBuilding.test_single_feature_dominance) ... ok

----------------------------------------------------------------------
Ran 6 tests in 0.099s

OK



All tests passed.


## **Part 5 — Tree Prediction & Navigation**

### **How the Tree Makes Predictions**

In this part, you'll implement how the trained decision tree predicts new samples. The tree uses its learned structure to follow paths from the **root** to the correct **leaf** node.

- **predict()** takes a dataset and returns a predicted label for each sample
- **_move_Tree()** moves through tree structure following the decision path
- The process begins at the root and continues until a leaf node is reached
- Each **leaf node** stores the final class label (or value) that becomes the prediction

*predict* and *_move_Tree* functions with TODOs are in [DT_Library](DT_Library.py) — complete them so that the tests check your functions navigate tree correctly and return accurate predictions.

In [48]:
# ============================================================================
# TESTS FOR TREE PREDICTION & NAVIGATION
# ============================================================================

class TestTreePrediction(unittest.TestCase):

    # ===============================
    # HELPER FUNCTIONS
    # ===============================

    def create_gini_tree(self):
        # Create a simple tree with clear class separation using Gini criterion
        Y_train = np.array([0, 0, 0, 2, 2, 1, 2, 2])
        X_train = pd.DataFrame({
            'feature1': [1, 1, 1, 2, 2, 2, 3, 3],
            'feature2': [9, 3, 7, 1, 5, 8, 2, 6],
            'feature3': [5, 5, 5, 5, 4, 5, 5, 5]
        })
        
        dt = DecisionTree(mode="gini", min_Samples=1, max_Depth=3)
        dt.fit(X_train, Y_train)
        return dt, X_train, Y_train

    def create_gain_tree(self):
        # Create a more complex tree using Information Gain criterion
        Y_train = np.array([0, 1, 2, 1, 1, 3, 1, 1, 3, 0, 1, 2])
        X_train = pd.DataFrame({
            "feature1": [1, 2, 2, 2, 2, 2, 3, 3, 3, 1, 4, 4],
            "feature2": [0, 1, 2, 0, 1, 3, 0, 1, 3, 0, 1, 1],
            'feature3': [0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1],
            'feature4': [0, 2, 0, 1, 1, 2, 1, 0, 0, 1, 2, 1]
        })
        
        dt = DecisionTree(mode="gain", min_Samples=2, max_Depth=2)
        dt.fit(X_train, Y_train)
        return dt, X_train, Y_train
    
    # ===============================
    # TEST FUNCTIONS
    # ===============================

    def test_predict_output_shape(self):
        # Ensure predict() returns a valid array with the same length as input
        dt, X_train, Y_train = self.create_gini_tree()
        
        predictions = dt.predict(X_train)
        
        self.assertIsNotNone(predictions, "predict() should not return None")
        self.assertEqual(len(predictions), len(X_train), "predict() should return one output per input sample")
        
    def test_prediction_accuracy_on_training_data(self):
        # Check that the model achieves high accuracy on the training set
        dt, X_train, Y_train = self.create_gain_tree()
        
        predictions = dt.predict(X_train)
        
        accuracy = np.mean(predictions == Y_train)
        self.assertAlmostEqual(
            accuracy, 
            0.91666666666, 
            places=3, 
            msg=f"Model should perform well on training data, got {accuracy:.2%}"
        )

    def test_predictions_use_correct_features_gain(self):
        # Verify that predictions are based on the most informative features (Information Gain)
        dt, X_train, Y_train = self.create_gain_tree()
        
        X_test = pd.DataFrame({
            'feature1': [1, 2, 2, 4],
            'feature2': [0, 0, 1, 3],
            'feature3': [0, 0, 0, 1],
            'feature4': [2, 2, 1, 0]
        })
        
        predictions = dt.predict(X_test)
        
        self.assertEqual(len(predictions), 4, "predict() should return a prediction for each test sample")
        
        for i, pred in enumerate(predictions):
            self.assertIn(pred, [0, 1, 3], f"Prediction {pred} should be one of the valid class labels")

        self.assertEqual(predictions[3], 3, "Sample 3 should be classified as class 3")

    def test_predictions_use_correct_features_gini(self):
        # Verify that predictions are based on key features (Gini criterion)
        dt, X_train, Y_train = self.create_gini_tree()
        
        X_test = pd.DataFrame({
            'feature1': [1, 3, 2],
            'feature2': [3, 2, 6],
            'feature3': [5, 4, 4]
        })
        
        predictions = dt.predict(X_test)
        
        self.assertEqual(len(predictions), 3, "predict() should return a prediction for each test sample")
        
        for i, pred in enumerate(predictions):
            self.assertIn(pred, [0, 2], f"Prediction {pred} should be one of the valid class labels")

        self.assertEqual(predictions[0], 0, "Sample 0 should be classified as class 0")

# Run the tests
runner = unittest.TextTestRunner(verbosity=2)
suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestTreePrediction)
result = runner.run(suite)

print("\nAll tests passed." if result.wasSuccessful() else "\nSome tests failed.")

test_predict_output_shape (__main__.TestTreePrediction.test_predict_output_shape) ... ok
test_prediction_accuracy_on_training_data (__main__.TestTreePrediction.test_prediction_accuracy_on_training_data) ... ok
test_predictions_use_correct_features_gain (__main__.TestTreePrediction.test_predictions_use_correct_features_gain) ... ok
test_predictions_use_correct_features_gini (__main__.TestTreePrediction.test_predictions_use_correct_features_gini) ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.039s

OK



All tests passed.


## **Part 6 — Hyperparameter Optimization**

### **How to Optimize and Control the Tree**

In this part, you'll implement advanced techniques to find the best tree settings and prevent overfitting. These methods help create trees that work well on new, unseen data.

- **Grid search** tries many combinations of hyperparameters and picks the combination that gives the best validation performance. Proper tuning can reduce both underfitting and overfitting. you can use random search too.

**Note:** There are no tests for this part — you must implement and validate these functions yourself.

**Validation tips** (keep these results for your presentation):

* **Grid search:** Try many hyperparameter combinations, print the validation accuracies for each, compare them, and identify whether the chosen hyperparameters look truly optimal.
* **Keep your results:** Keep the printed outputs and a short written analysis — you will be asked about these in your presentation.

In [49]:
# ============================================================================
# Hyperparameter Optimization & Post-Pruning
# ============================================================================

def calculate_accuracy(tree, X, y):
    predictions = tree.predict(X)
    return np.mean(predictions == y)

def grid_search(X_train, y_train, X_val, y_val):
    best_params = {}
    best_score = 0
    results = []
    max_depths= [6]
    min_samples= [30]
    # max_depths = [8]
    # min_samples = [25]
    modes = ['gini']
    
    for depth in max_depths:
        for min_sample in min_samples:
            for mode in modes:
                tree = DecisionTree(
                    mode=mode,
                    max_Depth=depth,
                    min_Samples=min_sample
                )
                
                tree.fit(X_train, y_train)
                accuracy = calculate_accuracy(tree, X_val, y_val)
                
                print(f"Depth: {depth}, Min Samples: {min_sample}, Mode: {mode} -> Accuracy: {accuracy}")
                
                results.append({
                    'max_depth': depth,
                    'min_samples': min_sample,
                    'mode': mode,
                    'accuracy': accuracy
                })
                
                if accuracy > best_score:
                    best_score = accuracy
                    best_params = {
                        'max_depth': depth,
                        'min_samples': min_sample,
                        'mode': mode
                    }
    
    return best_params, best_score, results

## **Part 7 — Data Cleaning & Preprocessing**

### **How to Prepare Real Data for Your Tree**

In this part, you'll **clean and preprocess the EuroRail_Survey dataset** to make it ready for decision tree modeling. Real-world data is often messy, incomplete, or inconsistent — proper preprocessing ensures reliable learning and accurate predictions.

* **Missing Values Handling:** Fill or remove incomplete records using smart strategies (e.g., median for numeric data or mode for categorical).
* **Duplicate Removal:** Detect and delete duplicate rows to avoid bias during training.
* **Outlier Detection:** Identify extreme or unrealistic values that could distort splits and model accuracy.
* **Feature Scaling:** Normalize or standardize numeric features (using *MinMaxScaler* or *StandardScaler*) to make feature comparisons fair.
* **Feature Selection:** Remove irrelevant or highly correlated attributes using correlation analysis or heatmaps to reduce redundancy.
* **Categorical Encoding:** Transform text-based features into numeric form with *LabelEncoder* or *OneHotEncoder*.
* **Binning:** Convert continuous variables into discrete bins to help the tree find clearer splitting patterns.

**Note:** Not all steps are mandatory — analyze your data and choose methods that improve accuracy and interpretability.

**Important:** Save your cleaned dataset and clearly document each preprocessing step you applied and why. Then, **compare model performance before and after preprocessing** to show how data preparation improved your decision tree.

In [50]:
# ============================================================================
# DATA CLEANING & PREPROCESSING
# ============================================================================

# Load the raw dataset
full_Data = pd.read_csv('EuroRail_Survey.csv')
# print('\nDataset Info:')
# print(full_Data.info(), '\n')
# print(full_Data.dtypes)
# print(full_Data.isnull().sum().sort_values(ascending=False))
# print(full_Data.select_dtypes(include=['object']).columns)
# num_duplicates = full_Data.duplicated().sum()
# print(f"Number of duplicate rows: {num_duplicates}")
# ============================================================================
# HELPER FUNCTIONS FOR PREPROCESSING
# ============================================================================
def dropuselesscolumns(df):
    if 'id' in df.columns:
        df = df.drop(columns=['id'])
    return df

def capoutliers(df, numeric_cols):
    for col in numeric_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        df[col] = df[col].clip(lower, upper)
    return df

def fillmissingnumeric(df):
    for col in df.select_dtypes(include=['float64', 'int64']).columns:
        df[col] = df[col].fillna(df[col].median())
    return df

def fillmissingcategorical(df):
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].fillna(df[col].mode()[0])
    return df

def removeduplicates(df):
    return df.drop_duplicates()


def encodecategoricallabels(df, cat_cols):
    for col in cat_cols:
        df[col] = df[col].astype(object)
        df[col] = df[col].fillna('MISSING')
        df[col] = pd.Categorical(df[col]).codes
    return df


# ============================================================================ 
#  FUNCTIONS
# ============================================================================ 
def smart_age_binning(df):
    bins = [0, 18, 30, 45, 60, 100]
    labels = ['Child_Teen', 'Young_Adult', 'Adult', 'Middle_Aged', 'Senior']
    df['Age_Group'] = pd.cut(df['Age'], bins=bins, labels=labels)
    return df

def smart_delay_binning(df):
    dep_bins = [-1, 0, 15, 60, 120, float('inf')]
    dep_labels = ['No_Delay', 'Short_Delay', 'Medium_Delay', 'Long_Delay', 'Very_Long_Delay']
    df['Departure_Delay_Level'] = pd.cut(df['Departure Delay in Minutes'], 
                                       bins=dep_bins, labels=dep_labels)

    arr_bins = [-1, 0, 15, 60, 120, float('inf')]
    arr_labels = ['No_Delay', 'Short_Delay', 'Medium_Delay', 'Long_Delay', 'Very_Long_Delay']
    df['Arrival_Delay_Level'] = pd.cut(df['Arrival Delay in Minutes'], 
                                     bins=arr_bins, labels=arr_labels)
    return df

def smart_distance_binning(df):
    q1 = df['Trip Distance'].quantile(0.25)
    q2 = df['Trip Distance'].quantile(0.5)
    q3 = df['Trip Distance'].quantile(0.75)
    bins = [0, q1, q2, q3, float('inf')]
    labels = ['Short', 'Medium', 'Long', 'Very_Long']
    df['Distance_Category'] = pd.cut(df['Trip Distance'], bins=bins, labels=labels)
    return df

def smart_train_age_binning(df):
    current_year = 2024
    df['Train_Age'] = current_year - df['Train Model Year']
    bins = [0, 10, 20, 30, 50, 100]
    labels = ['New', 'Young', 'Middle', 'Old', 'Very_Old']
    df['Train_Age_Group'] = pd.cut(df['Train_Age'], bins=bins, labels=labels)
    return df

# ============================================================================ 
# PIPELINE
# ============================================================================ 
full_Data = full_Data.dropna(subset=['satisfaction'])
full_Data = dropuselesscolumns(full_Data)

numeric_cols = [
    'Departure Delay in Minutes',
    'Arrival Delay in Minutes',
    'Ticket Verification Service',
    'Train Staff Service',
    'Trip Distance'
]


full_Data = fillmissingnumeric(full_Data)
full_Data = fillmissingcategorical(full_Data)
full_Data = removeduplicates(full_Data)


full_Data = smart_age_binning(full_Data)
full_Data = smart_delay_binning(full_Data)
full_Data = smart_distance_binning(full_Data)
full_Data = smart_train_age_binning(full_Data)


categorical_cols = [
    'Passenger Type', 'Type of Trip', 'Ticket Class', 'nation', 'satisfaction',
    'Age_Group', 'Departure_Delay_Level', 'Arrival_Delay_Level', 
    'Distance_Category', 'Train_Age_Group'
]
full_Data = encodecategoricallabels(full_Data, categorical_cols)
full_Data = capoutliers(full_Data, numeric_cols)
# ============================================================================ 
# Save cleaned dataset
# ============================================================================
full_Data.to_csv('EuroRail_Survey_cleaned.csv', index=False)
print("\n Cleaned Data Info:")
print(full_Data.info())
print('\nPreprocessing completed successfully!')





 Cleaned Data Info:
<class 'pandas.core.frame.DataFrame'>
Index: 129293 entries, 0 to 129879
Data columns (total 30 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   Passenger Type                     129293 non-null  int8   
 1   Age                                129293 non-null  float64
 2   Type of Trip                       129293 non-null  int8   
 3   Ticket Class                       129293 non-null  int8   
 4   Trip Distance                      129293 non-null  float64
 5   Seat comfort                       129293 non-null  float64
 6   Departure/Arrival time convenient  129293 non-null  float64
 7   Dining Service                     129293 non-null  float64
 8   Platform Location                  129293 non-null  float64
 9   Onboard wifi service               129293 non-null  float64
 10  Onboard entertainment              129293 non-null  float64
 11  Online support         

## **Part 8 — Model Training & Testing**

### **Train Your Decision Tree on Clean Data**

Use your cleaned dataset to train and test your DecisionTree class. Calculate and display both **training and test accuracy**.

**Note:** If you achieve over 90% accuracy on test data, document your analysis explaining how you reached this performance level, and you'll receive bonus points.

In [51]:
# ============================================================================
# MODEL TRAINING & TESTING
# ============================================================================
# Load cleaned dataset
full_Data = pd.read_csv('EuroRail_Survey_cleaned.csv')
# Split features and target variable
X_Data = full_Data.drop(columns=['satisfaction'])
Y_Data = full_Data['satisfaction']
# Split data into training+validation and testing sets
X_TrainVal, X_Test, Y_TrainVal, Y_Test = train_test_split(
    X_Data, Y_Data, test_size=0.2, random_state=42)
# Further split training data for validation
X_Train, X_Val, Y_Train, Y_Val = train_test_split(
    X_TrainVal, Y_TrainVal, test_size=0.2, random_state=42)
# Grid search to find best hyperparameters
best_params, best_score, results = grid_search(X_Train, Y_Train, X_Val, Y_Val)
print("\nBest Hyperparameters from Grid Search:", best_params)
print("Best Validation Accuracy from Grid Search:", best_score)
# Train final model on full training+validation set with best hyperparameters
final_tree = DecisionTree(
    mode=best_params['mode'],
    max_Depth=best_params['max_depth'],
    min_Samples=best_params['min_samples']
)
# final_tree = DecisionTree(
#     mode='gini',
#     max_Depth=7,
#     min_Samples=20
# )
final_tree.fit(X_TrainVal, Y_TrainVal)
# Make predictions
train_pred = final_tree.predict(X_TrainVal)
test_pred = final_tree.predict(X_Test)
val_pred = final_tree.predict(X_Val)
# Calculate accuracies
train_accuracy = np.mean(train_pred == Y_TrainVal)
test_accuracy = np.mean(test_pred == Y_Test)
val_accuracy = np.mean(val_pred == Y_Val)
print('\nTrain Accuracy:', train_accuracy)
print('Validation Accuracy:', val_accuracy)
print('Test Accuracy:', test_accuracy)

print('\nModel training and testing completed!')

Depth: 6, Min Samples: 30, Mode: gini -> Accuracy: 0.8093005269009523

Best Hyperparameters from Grid Search: {'max_depth': 6, 'min_samples': 30, 'mode': 'gini'}
Best Validation Accuracy from Grid Search: 0.8093005269009523

Train Accuracy: 0.9220759131426803
Validation Accuracy: 0.9237202107603809
Test Accuracy: 0.8340229707258595

Model training and testing completed!


## **Part 9 — Post-Pruning in Decision Trees**

### **What is Post-Pruning?**

Post-pruning (also called "cost-complexity pruning") is a technique used to reduce overfitting in decision trees.  
After the tree is fully grown (possibly overfitting the training data), some branches are removed to make the tree simpler and improve its performance on unseen data.

**Why use it?**

- Fully grown trees can memorize noise in the training data.  
- Post-pruning simplifies the tree, making it more generalizable.  

**How to do it (briefly):**

1. Grow the full tree using your training data.  
2. Evaluate the performance of each subtree on a **validation set** (or using cross-validation).  
3. Remove branches/subtrees that do not improve performance on the validation data.  
4. Repeat until removing any branch would decrease validation accuracy.  

- **NOTE:** Post-pruning is applied **after the tree is fully built**, unlike pre-pruning which stops the tree growth early.

**Validation tips** (keep these results for your presentation):

* **Post-pruning:** Show the tree before and after pruning (a plot), report validation/test accuracy numbers, and explain if pruning improved generalization and why.
* **Keep your results:** Keep the printed outputs and a short written analysis — you will be asked about these in your presentation.

## **Part 10 — Decision Tree Visualization**

### **Why visualize a decision tree?**  
Visualizing a decision tree helps you understand how the tree splits the data, which features are important, and how decisions are made at each node.

**How to visualize a tree:**  

You can use several Python tools to draw your decision tree:

1. **Graphviz**  
   - Popular for creating clean, professional tree diagrams.
   - Can export as PDF, PNG, or SVG.

2. **NetworkX**  
   - Can build a tree as a graph and visualize nodes and edges.  
   - Allows more flexible customizations, colors, and layouts.  

3. **Other libraries**  
   - *matplotlib* with custom plotting  
   - *pydotplus* (works with Graphviz)  

**NOTE:**  
- Always try to include feature names and class labels in the visualization.  
- Use validation or pruning to avoid overly large trees that are hard to read.  
- For complex trees, exporting as PDF or using zoomable interactive plots is recommended.


In [53]:

def visualize_tree_smart(root, max_children_display=5, filename="decision_tree"):

    dot = Digraph(comment='Decision Tree')
    dot.attr('node', shape='box', style='rounded', fontname='Arial', fontsize='10')
    dot.attr('edge', fontname='Arial', fontsize='8')
    dot.attr(bgcolor="white")

    def add_node(node, parent_id=None, edge_label=""):
        node_id = str(id(node))
        
        if node.children and len(node.children) > max_children_display:
            label = f"{node.feature}\n({len(node.children)} children)"
            if node.value is not None:
                label += f"\nvalue={node.value}"
            dot.node(node_id, label, fillcolor='orange', style='filled')
            
            for i, child in enumerate(node.children[:max_children_display]):
                if hasattr(child, 'feature_value') and child.feature_value is not None:
                    child_edge_label = str(child.feature_value)
                else:
                    child_edge_label = ""
                add_node(child, node_id, child_edge_label)
            
            remaining_id = f"{node_id}_remaining"
            dot.node(remaining_id, f"... {len(node.children) - max_children_display} more",
                     fillcolor='lightgray', style='filled')
            dot.edge(node_id, remaining_id, label="...")

        elif node.children:
            label = f"{node.feature}"
            if node.value is not None:
                label += f"\n(value={node.value})"
            dot.node(node_id, label, fillcolor='lightblue', style='filled')
            
            for child in node.children:
                if hasattr(child, 'feature_value') and child.feature_value is not None:
                    child_edge_label = str(child.feature_value)
                else:
                    child_edge_label = ""
                add_node(child, node_id, child_edge_label)
        
        else:
            label = f"Class: {node.value}"
            dot.node(node_id, label, fillcolor='lightgreen', style='filled')
        
        if parent_id:
            dot.edge(parent_id, node_id, label=edge_label)

    add_node(root)
    dot.render(filename, format='png', cleanup=True)
    print(f"Tree visualization saved as {filename}.png")
    return dot
dot = visualize_tree_smart(final_tree.root, max_children_display=2, filename="final_tree_visual")

Tree visualization saved as final_tree_visual.png
